# scikit-learn 学习笔记（入门速通）

> 目标：掌握 sklearn 的统一 API、数据划分、预处理、常用监督/无监督模型、评估调参与 Pipeline。

scikit-learn 是 Python 机器学习的核心库，导入名为 `sklearn`。它基于 NumPy 和 SciPy 构建，提供了数十种经典机器学习算法的**一致接口**：所有模型都是 `Estimator`，通过 `fit(X, y)` 训练，`predict(X)` 预测；预处理器则用 `transform(X)` 转换数据[citation:4]。这种统一性是 sklearn 最大的价值——学会一个模型，就学会了所有模型的用法。

**环境准备**
```bash
pip install scikit-learn
```

> 注意：包名是 `scikit-learn`（带连字符），代码中导入 `sklearn`，不是 `scikit`。

In [1]:
import sklearn
import numpy as np
print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.7.1


## 1. 统一 API 范式

sklearn 的所有对象共享一套简洁的接口，理解这三类接口就掌握了库的骨架[citation:4][citation:8]：

| 接口 | 方法 | 用途 |
|---|---|---|
| Estimator | `fit(X, y)` | 训练模型 |
| Predictor | `predict(X)` | 对新数据预测 |
| Transformer | `transform(X)` | 数据转换（预处理/降维） |

**数据约定**：`X` 是形状为 `(n_samples, n_features)` 的二维数组（样本 × 特征），`y` 是形状 `(n_samples,)` 的目标值。X 和 y 通常是 NumPy 数组或类似结构[citation:4]。

**标准工作流**
1. 准备数据 → `train_test_split` 划分
2. 预处理 → `StandardScaler` 等（`fit_transform` 训练集，`transform` 测试集）
3. 训练 → `model.fit(X_train, y_train)`
4. 预测 → `model.predict(X_test)`
5. 评估 → `accuracy_score` / `r2_score` 等

**易错点**：预处理器的 `fit` **只能**在训练集上调用，测试集只调用 `transform`，否则会造成数据泄露。

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 完整工作流演示
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # 训练集：fit + transform
X_test_s = scaler.transform(X_test)         # 测试集：只 transform

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_s, y_train)
pred = clf.predict(X_test_s)
print("准确率:", accuracy_score(y_test, pred))

准确率: 1.0


## 2. 数据加载与划分

sklearn 内置了多个玩具数据集，也支持生成合成数据。

| 函数 | 用途 |
|---|---|
| `load_iris()` / `load_digits()` / `load_wine()` | 内置经典数据集 |
| `make_classification()` / `make_regression()` | 生成合成数据 |
| `train_test_split(X, y, test_size, random_state)` | 划分训练/测试集 |

**`train_test_split` 关键参数**：`test_size` 控制测试集比例（默认 0.25），`random_state` 固定随机种子保证可复现，`stratify=y` 保持类别比例一致（分类任务推荐）。

**交叉验证**：`cross_val_score(model, X, y, cv=5)` 做 K 折交叉验证，比单次划分更可靠地评估模型泛化能力[citation:4]。

In [3]:
from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression

# 生成合成数据
X, y = make_classification(n_samples=200, n_features=10, n_classes=2, random_state=42)
print("数据形状:", X.shape, y.shape)

# 分层划分
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print("训练集:", X_tr.shape, "测试集:", X_te.shape)

# 交叉验证
iris_X, iris_y = load_iris(return_X_y=True)
scores = cross_val_score(LogisticRegression(max_iter=200), iris_X, iris_y, cv=5)
print("5折交叉验证准确率:", scores.round(3), "均值:", scores.mean().round(3))

数据形状: (200, 10) (200,)
训练集: (140, 10) 测试集: (60, 10)
5折交叉验证准确率: [0.967 1.    0.933 0.967 1.   ] 均值: 0.973


## 3. 数据预处理

预处理器是 Transformer，用 `fit_transform` 训练集、`transform` 测试集。

| 类 | 作用 | 适用场景 |
|---|---|---|
| `StandardScaler` | 标准化（均值 0，方差 1） | 大多数模型，尤其是 SVM、线性模型 |
| `MinMaxScaler` | 缩放到 [0, 1] | 神经网络、需要固定范围的场景 |
| `OneHotEncoder` | 类别变量独热编码 | 无序类别特征 |
| `SimpleImputer` | 缺失值填充 | 用均值/中位数/众数填补 |

**`ColumnTransformer`**：当不同列需要不同预处理时，用它组合多个转换器，指定每列应用哪个处理器[citation:4]。

**易错点**：`StandardScaler` 的 `fit` 会计算训练集的均值和标准差，测试集必须复用这些参数（即只调 `transform`），否则标准化不一致。

In [4]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import numpy as np

X = np.array([[1.0, 2.0], [2.0, np.nan], [3.0, 6.0], [4.0, 8.0]])

# 缺失值填充
imputer = SimpleImputer(strategy='mean')
X_filled = imputer.fit_transform(X)
print("填充后:\n", X_filled)

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)
print("标准化后均值:", X_scaled.mean(axis=0).round(6))

# 独热编码
enc = OneHotEncoder(sparse_output=False)
cat = enc.fit_transform([['红'], ['绿'], ['红'], ['蓝']])
print("独热编码:\n", cat)

填充后:
 [[1.         2.        ]
 [2.         5.33333333]
 [3.         6.        ]
 [4.         8.        ]]
标准化后均值: [0. 0.]
独热编码:
 [[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]


## 4. 监督学习：分类与回归

sklearn 的监督学习模型全部遵循 `fit` / `predict` 接口。

**常用分类器**

| 类 | 算法 | 特点 |
|---|---|---|
| `LogisticRegression` | 逻辑回归 | 线性、可解释性强、速度快 |
| `RandomForestClassifier` | 随机森林 | 集成方法、鲁棒、无需过多调参 |
| `SVC` | 支持向量机 | 高维有效、需标准化 |
| `KNeighborsClassifier` | K 近邻 | 简单、惰性学习 |

**常用回归器**：`LinearRegression`、`Ridge`、`RandomForestRegressor`、`GradientBoostingRegressor`。

**选择建议**：先试 `RandomForestClassifier` 建立 baseline，再根据数据规模和可解释性需求尝试其他模型。

In [ ]:
from sklearn.datasets import load_iris, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score

# 分类
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
clf = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)
print("分类准确率:", accuracy_score(y_te, clf.predict(X_te)).round(3))

# 回归
Xd, yd = load_diabetes(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, random_state=0)
reg = RandomForestRegressor(n_estimators=100, random_state=0).fit(Xd_tr, yd_tr)
print("回归 R²:", r2_score(yd_te, reg.predict(Xd_te)).round(3))

## 5. 无监督学习：聚类与降维

无监督方法不需要标签 `y`，核心是发现数据内在结构[citation:5]。

**聚类**

| 类 | 算法 | 特点 |
|---|---|---|
| `KMeans` | K 均值 | 需指定簇数 K，速度快 |
| `DBSCAN` | 密度聚类 | 自动发现簇数，可识别噪声 |
| `AgglomerativeClustering` | 层次聚类 | 无需预设 K，可看树状图 |

**降维**

| 类 | 算法 | 用途 |
|---|---|---|
| `PCA` | 主成分分析 | 线性降维、可视化 |
| `TSNE` | t-SNE | 高维数据二维可视化（仅用于可视化，不用于建模） |

**易错点**：`KMeans` 的 `n_clusters` 必须手动指定；`TSNE` 计算较慢且结果随机，务必设 `random_state`。

In [ ]:
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

X, y = load_iris(return_X_y=True)

# KMeans 聚类
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
print("聚类标签分布:", np.bincount(labels))

# PCA 降维
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
print("降维后形状:", X_pca.shape)
print("解释方差比:", pca.explained_variance_ratio_.round(3))

## 6. 模型评估与超参数调优

**评估指标**

| 任务 | 常用函数 |
|---|---|
| 分类 | `accuracy_score`、`classification_report`、`confusion_matrix` |
| 回归 | `r2_score`、`mean_squared_error`、`mean_absolute_error` |

**超参数搜索**：`GridSearchCV` 穷举参数组合，`RandomizedSearchCV` 随机采样。两者都在交叉验证基础上搜索最优参数[citation:1][citation:4]。

**`GridSearchCV` 关键参数**：`param_grid`（参数字典）、`cv`（折数）、`scoring`（评估指标）、`n_jobs=-1`（并行）。

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report

X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None]
}
grid = GridSearchCV(RandomForestClassifier(random_state=0), param_grid, cv=3, n_jobs=-1)
grid.fit(X_tr, y_tr)
print("最优参数:", grid.best_params_)
print("最优交叉验证分:", grid.best_score_.round(3))

best = grid.best_estimator_
print("\n测试集报告:")
print(classification_report(y_te, best.predict(X_te)))

## 7. Pipeline 管道

`Pipeline` 把预处理和模型串联成一个对象，对外暴露与普通 Estimator 完全相同的 `fit` / `predict` 接口[citation:4]。

**核心价值**
1. **防止数据泄露**：交叉验证时，预处理只在每折的训练部分 fit，不会“看到”验证数据。
2. **代码简洁**：一个对象完成预处理 + 建模 + 预测。
3. **便于调参**：`GridSearchCV` 可以直接搜索管道中任意步骤的参数（用 `步骤名__参数名`）。

**构造方式**：`Pipeline([('名称', 对象), ...])`，最后一步必须是 Estimator（有 `predict`），前面的步骤是 Transformer。

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV

X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC())
])

param_grid = {'svc__C': [0.1, 1, 10], 'svc__kernel': ['rbf', 'linear']}
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_tr, y_tr)
print("最优参数:", grid.best_params_)
print("测试集准确率:", grid.score(X_te, y_te).round(3))

## 8. 综合练习

1. 加载 `load_wine` 数据集，用 `Pipeline` 组合 `StandardScaler` + `LogisticRegression`，做 5 折交叉验证并打印均值。
2. 对 `load_digits` 用 `KMeans(n_clusters=10)` 聚类，与真实标签对比（提示：聚类标签与真实标签编号不对应，需用 `adjusted_rand_score` 评估）。
3. 用 `GridSearchCV` 在 `RandomForestRegressor` 上搜索 `n_estimators` 和 `max_depth`，数据集用 `load_diabetes`。

**参考答案**

In [ ]:
# 1. Pipeline + 交叉验证
from sklearn.datasets import load_wine, load_digits, load_diabetes
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.ensemble import RandomForestRegressor

Xw, yw = load_wine(return_X_y=True)
pipe = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(max_iter=500))])
scores = cross_val_score(pipe, Xw, yw, cv=5)
print("1. 交叉验证均值:", scores.mean().round(3))

# 2. KMeans + ARI
Xd, yd = load_digits(return_X_y=True)
km = KMeans(n_clusters=10, random_state=42, n_init=10)
labels = km.fit_predict(Xd)
print("2. ARI:", adjusted_rand_score(yd, labels).round(3))

# 3. GridSearchCV 回归
Xr, yr = load_diabetes(return_X_y=True)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=0)
grid = GridSearchCV(RandomForestRegressor(random_state=0), 
                    {'n_estimators': [50, 100], 'max_depth': [3, 5]}, cv=3)
grid.fit(Xr_tr, yr_tr)
print("3. 最优:", grid.best_params_, "R²:", grid.score(Xr_te, yr_te).round(3))

## 常见坑点总结

- **包名 `scikit-learn`，导入 `sklearn`**：不是 `import scikit`，也不是 `import scikit-learn`。
- **预处理只在训练集 `fit`**：测试集用 `transform`，否则数据泄露导致评估虚高。
- **`X` 必须是二维**：sklearn 要求形状 `(n_samples, n_features)`，单特征也要 `reshape(-1, 1)`。
- **`random_state` 要固定**：涉及随机性的模型/划分，不设种子结果不可复现。
- **`Pipeline` 调参用 `步骤名__参数`**：双下划线，如 `svc__C`。
- **`KMeans` 必须指定 `n_clusters`**：且初始化随机，建议设 `random_state` 和 `n_init`。
- **`TSNE` 只用于可视化**：降维结果不能直接用于后续建模（无 `transform` 方法）。
- **先跑 baseline 再优化**：`RandomForest` / `LogisticRegression` 通常能快速给出可接受的起点。